# Trust-region methods

A trust-region method builds a quadratic model of the objective and minimizes
it inside a ball whose radius it grows or shrinks according to how well the
model predicted the last step. `TrustRegion` takes the subproblem solver as an
argument: `cauchy` (steepest descent inside the ball) or `dogleg` (a blend of
the steepest-descent and Newton steps).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import NLPProblem, TrustRegion, cauchy, dogleg

## Test problem

A quartic built from two positive definite matrices,

$$f(x) = (x^T Q x + b^T x)^2 + x^T M x + c^T x,$$

which has no closed-form minimizer, so results are checked against
`scipy.optimize.minimize(method="trust-exact")`.

In [2]:
Q = np.array([
    [1.85, 0.03, 0.00, 0.52, 0.05],
    [0.03, 1.71, -0.22, -0.32, -0.14],
    [0.00, -0.22, 1.75, 0.14, -0.12],
    [0.52, -0.32, 0.14, 1.45, 0.18],
    [0.05, -0.14, -0.12, 0.18, 1.91],
])
M = np.array([
    [3.08, 1.28, -0.14, -0.40, 0.38],
    [1.28, 5.06, 0.91, 0.86, 0.89],
    [-0.14, 0.91, 4.77, 0.61, 0.15],
    [-0.40, 0.86, 0.61, 3.28, 1.59],
    [0.38, 0.89, 0.15, 1.59, 3.99],
])
b = np.array([1.6, 1.2, 0.9, 0.6, -0.1])
c = np.array([-0.2, 0.8, 1.4, -0.7, -0.3])

s = lambda x: x @ Q @ x + b @ x
f = lambda x: float(s(x) ** 2 + x @ M @ x + c @ x)
grad_f = lambda x: 2.0 * s(x) * (2.0 * (Q @ x) + b) + 2.0 * (M @ x) + c

def hess_f(x):
    g_s = 2.0 * (Q @ x) + b
    return 2.0 * np.outer(g_s, g_s) + 4.0 * s(x) * Q + 2.0 * M

print("eigenvalues of the Hessian at 0:", np.round(np.linalg.eigvalsh(hess_f(np.zeros(5))), 3))

eigenvalues of the Hessian at 0: [ 4.013  5.361  9.126  9.999 22.22 ]


## Both subproblem solvers, across starting radii

The initial radius is a solver argument, not a property of the problem, so it
is worth seeing how much it matters.

In [3]:
rng = np.random.default_rng(42)
inits = np.vstack([np.zeros(5), rng.uniform(-2.0, 2.0, size=(3, 5))])
radii = [5.0, 1.0, 0.1, 0.01]

rows = []
for radius in radii:
    for x0 in inits:
        problem = NLPProblem(f=f, x0=x0, grad=grad_f, hess=hess_f)
        x_ref = minimize(f, x0, jac=grad_f, hess=hess_f, method="trust-exact").x
        for method in (cauchy, dogleg):
            result = TrustRegion(method=method, radius=radius).solve(problem)
            assert result.success, result.message
            np.testing.assert_allclose(result.x, x_ref, atol=1e-5)
            rows.append({
                "radius": radius,
                "x0": tuple(np.round(x0, 3)),
                "method": method.__name__,
                "iters": result.n_iter,
                "f": result.fun,
                "|x - x_scipy|": np.linalg.norm(result.x - x_ref),
            })

table = (pd.DataFrame(rows)
         .pivot(index=["radius", "x0"], columns="method")
         .swaplevel(axis=1)
         .reindex(columns=pd.MultiIndex.from_product(
             [["cauchy", "dogleg"], ["iters", "f", "|x - x_scipy|"]])))
table

cauchy                          \
                                              iters         f |x - x_scipy|   
radius x0                                                                     
0.01   (-0.517, 1.707, 0.575, 1.291, -0.226)     41 -0.209272  9.801085e-08   
       (0.0, 0.0, 0.0, 0.0, 0.0)                 30 -0.209272  1.454539e-07   
       (1.096, -0.244, 1.434, 0.789, -1.623)     39 -0.209272  8.456380e-08   
       (1.902, 1.045, 1.144, -1.488, -0.198)     40 -0.209272  7.408332e-07   
0.10   (-0.517, 1.707, 0.575, 1.291, -0.226)     28 -0.209272  7.803986e-08   
       (0.0, 0.0, 0.0, 0.0, 0.0)                 23 -0.209272  1.823024e-07   
       (1.096, -0.244, 1.434, 0.789, -1.623)     31 -0.209272  9.954136e-08   
       (1.902, 1.045, 1.144, -1.488, -0.198)     34 -0.209272  7.435288e-07   
1.00   (-0.517, 1.707, 0.575, 1.291, -0.226)     30 -0.209272  1.255387e-07   
       (0.0, 0.0, 0.0, 0.0, 0.0)                 31 -0.209272  1.625636e-07   
       (1.096, -0.244, 1.434, 0.789, -1.623)     36 -0.209272  9.414322e-08   
       (1.902, 1.045, 1.144, -1.488, -0.198)     30 -0.209272  7.544382e-07   
5.00   (-0.517, 1.707, 0.575, 1.291, -0.226)     30 -0.209272  1.255387e-07   
       (0.0, 0.0, 0.0, 0.0, 0.0)                 31 -0.209272  1.625636e-07   
       (1.096, -0.244, 1.434, 0.789, -1.623)     36 -0.209272  9.414322e-08   
       (1.902, 1.045, 1.144, -1.488, -0.198)     24 -0.209272  7.542342e-07   

                                             dogleg                          
                                              iters         f |x - x_scipy|  
radius x0                                                                    
0.01   (-0.517, 1.707, 0.575, 1.291, -0.226)     13 -0.209272  2.259838e-09  
       (0.0, 0.0, 0.0, 0.0, 0.0)                  8 -0.209272  4.338975e-09  
       (1.096, -0.244, 1.434, 0.789, -1.623)     14 -0.209272  2.803784e-08  
       (1.902, 1.045, 1.144, -1.488, -0.198)     14 -0.209272  7.333900e-07  
0.10   (-0.517, 1.707, 0.575, 1.291, -0.226)      9 -0.209272  8.053970e-08  
       (0.0, 0.0, 0.0, 0.0, 0.0)                  5 -0.209272  2.580880e-09  
       (1.096, -0.244, 1.434, 0.789, -1.623)     11 -0.209272  2.813990e-08  
       (1.902, 1.045, 1.144, -1.488, -0.198)     10 -0.209272  7.308612e-07  
1.00   (-0.517, 1.707, 0.575, 1.291, -0.226)      7 -0.209272  3.035766e-18  
       (0.0, 0.0, 0.0, 0.0, 0.0)                  4 -0.209272  2.792110e-17  
       (1.096, -0.244, 1.434, 0.789, -1.623)      8 -0.209272  4.009614e-17  
       (1.902, 1.045, 1.144, -1.488, -0.198)      8 -0.209272  7.333885e-07  
5.00   (-0.517, 1.707, 0.575, 1.291, -0.226)      7 -0.209272  3.035766e-18  
       (0.0, 0.0, 0.0, 0.0, 0.0)                  4 -0.209272  2.792110e-17  
       (1.096, -0.244, 1.434, 0.789, -1.623)      8 -0.209272  4.009614e-17  
       (1.902, 1.045, 1.144, -1.488, -0.198)      8 -0.209272  7.333900e-07

Every run agrees with SciPy. The interesting column is `iters`: `dogleg` uses
the Newton step whenever it fits inside the radius, so it converges in a
handful of iterations, while `cauchy` only ever moves along the negative
gradient and behaves like steepest descent.

In [4]:
means = pd.DataFrame(rows).groupby(["method", "radius"])["iters"].mean().unstack()
print("mean iterations by method and starting radius\n")
print(means.round(1))

mean iterations by method and starting radius

radius  0.01  0.10  1.00  5.00
method                        
cauchy  37.5  29.0  31.8  30.2
dogleg  12.2   8.8   6.8   6.8


## Failure is reported, not raised

`dogleg` needs a positive definite model. On a saddle it says so, and the
result object carries the reason rather than an exception escaping the solver.

In [5]:
saddle = NLPProblem(
    f=lambda x: float(x[0]**2 - x[1]**2),
    x0=[1.0, 1.0],
    grad=lambda x: np.array([2.0 * x[0], -2.0 * x[1]]),
    hess=lambda x: np.diag([2.0, -2.0]),
)
result = TrustRegion(method=dogleg).solve(saddle)
print("success:", result.success)
print("message:", result.message)

success: False
message: Subproblem failed: dogleg requires positive curvature along the gradient (positive definite model matrix)
